# Train RandLA-Net on USGS 3DEP Aerial LiDAR

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/opengeos/geoai/blob/main/docs/examples/train_point_cloud_3dep.ipynb)

This notebook trains a RandLA-Net model on freely available
[USGS 3DEP](https://www.usgs.gov/3d-elevation-program) classified aerial
LiDAR data and uploads the checkpoint to Hugging Face Hub.

The 3DEP program provides billions of classified LiDAR points covering the
entire US. We download diverse tiles from multiple regions (urban, suburban,
rural, coastal) and use the existing ASPRS ground-truth labels for training.

**Classes (7):** unclassified, ground, vegetation, building, water, bridge, noise

**Requirements:** GPU runtime on Google Colab (T4 or better).

## 1. Install dependencies

In [ ]:
# Colab: install compatible PyTorch first
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121

# Install geoai from the PR branch (has RandLANet_3DEP config)
!pip install "geoai-py[pointcloud] @ git+https://github.com/jayakumarpujar/geoai-1.git@feature/point-cloud-classification"

# These must come last to avoid being overwritten
!pip install "numpy<2" huggingface_hub

## 2. Download 3DEP LiDAR tiles

We use `leafmap.download_tnm()` to download classified LAZ tiles from the
USGS 3DEP program. We select diverse regions across the US to ensure the
model generalises well:

| Region | Description | Bbox (minx, miny, maxx, maxy) |
|--------|-------------|-------------------------------|
| Madison, WI | Urban/suburban, buildings + vegetation | [-89.42, 43.06, -89.37, 43.09] |
| Houston, TX | Flat urban, buildings + roads | [-95.40, 29.74, -95.35, 29.77] |
| Denver, CO | Urban + terrain variation | [-104.99, 39.73, -104.94, 39.76] |
| Portland, OR | Vegetation-heavy, bridges | [-122.68, 45.50, -122.63, 45.53] |
| Tampa, FL | Coastal, water + buildings | [-82.47, 27.94, -82.42, 27.97] |

Each tile contains ASPRS-classified points (ground, vegetation, buildings,
water, etc.) that serve as ground-truth labels for training.

In [ ]:
import os
from pathlib import Path

import leafmap

# Output directories
DATA_DIR = "/content/3dep_lidar"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
SAVE_DIR = "/content/checkpoints"

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

# Diverse regions across the US (training set)
TRAIN_REGIONS = {
    "madison_wi": [-89.42, 43.06, -89.37, 43.09],
    "houston_tx": [-95.40, 29.74, -95.35, 29.77],
    "denver_co": [-104.99, 39.73, -104.94, 39.76],
    "portland_or": [-122.68, 45.50, -122.63, 45.53],
}

# Held-out region for validation
VAL_REGIONS = {
    "tampa_fl": [-82.47, 27.94, -82.42, 27.97],
}

TNM_API = {
    "datasets": "Lidar Point Cloud (LPC)",
    "prodFormats": "LAZ",
    "max": 5,  # max tiles per region
}

for name, bbox in TRAIN_REGIONS.items():
    print(f"Downloading {name}...")
    leafmap.download_tnm(
        region=bbox,
        out_dir=TRAIN_DIR,
        API=TNM_API,
    )

for name, bbox in VAL_REGIONS.items():
    print(f"Downloading {name} (validation)...")
    leafmap.download_tnm(
        region=bbox,
        out_dir=VAL_DIR,
        API=TNM_API,
    )

train_files = sorted(Path(TRAIN_DIR).glob("*.la[sz]"))
val_files = sorted(Path(VAL_DIR).glob("*.la[sz]"))
print(f"\nTrain files: {len(train_files)}, Val files: {len(val_files)}")

## 3. Inspect downloaded data

Verify that the downloaded tiles contain diverse ASPRS classification codes.
The 3DEP model remaps these to 7 contiguous classes during training.

In [ ]:
import laspy
import numpy as np

from geoai.pointcloud import ASPRS_CLASSES, SUPPORTED_MODELS

asprs_to_model = SUPPORTED_MODELS["RandLANet_3DEP"]["asprs_to_model"]
class_names = SUPPORTED_MODELS["RandLANet_3DEP"]["class_names"]

print("ASPRS code -> Model class mapping:")
for asprs_code, model_idx in sorted(asprs_to_model.items()):
    print(
        f"  ASPRS {asprs_code:2d} ({ASPRS_CLASSES.get(asprs_code, '?'):20s}) -> {model_idx} ({class_names[model_idx]})"
    )

print(f"\n{'='*60}")
print("Scanning downloaded tiles...\n")

for las_path in sorted(Path(TRAIN_DIR).glob("*.la[sz]"))[:3]:
    las = laspy.read(str(las_path))
    codes, counts = np.unique(las.classification, return_counts=True)
    total = len(las.classification)
    print(f"{las_path.name}: {total:,} points")
    for code, count in zip(codes, counts):
        name = ASPRS_CLASSES.get(int(code), f"Code {code}")
        pct = count / total * 100
        print(f"  ASPRS {code:3d} ({name:20s}): {count:>10,} ({pct:.1f}%)")
    print()

## 4. Initialize model and train

We initialise a `PointCloudClassifier` with the `RandLANet_3DEP` config.
Since no pre-trained checkpoint exists yet, we use the SemanticKITTI weights
as a starting point for transfer learning (the architecture is overridden
by the 3DEP config).

**Note:** The first run will train from scratch since the 3DEP checkpoint
URL does not exist yet. After uploading the trained checkpoint (step 6),
future users will get the pre-trained weights automatically.

In [ ]:
from geoai.pointcloud import PointCloudClassifier

# For the initial training run, we need to provide a checkpoint_path
# manually since the 3DEP checkpoint hasn't been uploaded yet.
# Download SemanticKITTI weights as architecture initialisation:
from geoai.pointcloud import _download_checkpoint

base_ckpt = _download_checkpoint("RandLANet_SemanticKITTI")

classifier = PointCloudClassifier(
    model_name="RandLANet_3DEP",
    checkpoint_path=base_ckpt,  # transfer learning starting point
    device="cuda",
    num_classes=7,
)

In [ ]:
# Train - this will take several hours on a T4 GPU
history = classifier.train(
    train_dir=TRAIN_DIR,
    val_dir=VAL_DIR,
    epochs=100,
    learning_rate=0.001,
    batch_size=4,
    save_dir=SAVE_DIR,
)
print(f"Checkpoint saved: {history['checkpoint_path']}")

## 5. Test inference on a validation file

Run the trained model on a held-out validation tile to check results.
The output LAS file uses standard ASPRS codes (2=Ground, 6=Building, etc.)
for interoperability with tools like CloudCompare.

In [ ]:
test_file = str(val_files[0])
output_file = "/content/test_classified_3dep.las"

labels, probs = classifier.classify(test_file, output_path=output_file)
print(f"Classified {len(labels)} points into {len(np.unique(labels))} classes")

stats = classifier.summary(output_file)
print(f"\nClass distribution:")
for name, pct in stats["class_percentages"].items():
    count = stats["class_counts"][name]
    print(f"  {name:30s}: {count:>10,} ({pct:.1f}%)")

## 6. Upload checkpoint to Hugging Face Hub

Upload the trained checkpoint so it can be downloaded by `geoai`.
You'll need a [Hugging Face token](https://huggingface.co/settings/tokens).

In [ ]:
import hashlib

from huggingface_hub import HfApi, login

# Authenticate (paste your HF token when prompted)
login()

checkpoint_path = history["checkpoint_path"]

# Compute SHA-256 for integrity verification
h = hashlib.sha256()
with open(checkpoint_path, "rb") as f:
    for chunk in iter(lambda: f.read(65536), b""):
        h.update(chunk)
sha256 = h.hexdigest()
print(f"SHA-256: {sha256}")
print(f"Update SUPPORTED_MODELS['RandLANet_3DEP']['sha256'] with this value.")

# Upload to Hugging Face
api = HfApi()
api.create_repo("jayakumarpujar/randlanet-3dep", repo_type="model", exist_ok=True)
api.upload_file(
    path_or_fileobj=checkpoint_path,
    path_in_repo="randlanet_3dep.pth",
    repo_id="jayakumarpujar/randlanet-3dep",
)
print("Checkpoint uploaded to: https://huggingface.co/jayakumarpujar/randlanet-3dep")

## 7. Next steps

After uploading the checkpoint:

1. Copy the SHA-256 hash printed above
2. Update `SUPPORTED_MODELS["RandLANet_3DEP"]["sha256"]` in `geoai/pointcloud.py`
3. Push the update to the PR branch
4. Users can now run:

```python
from geoai.pointcloud import classify_point_cloud

labels, probs = classify_point_cloud(
    "my_aerial_lidar.las",
    output_path="classified.las",
    model_name="RandLANet_3DEP",
)
```

The output uses standard ASPRS codes (2=Ground, 5=High Vegetation,
6=Building, 9=Water, 17=Bridge, 7=Noise) for compatibility with
CloudCompare, QGIS, and other LiDAR tools.